# Vegetable Image Classification using CNN
**Innotech Deep Learning Project**

Run this notebook in Kaggle. Attach the Kaggle Vegetable Image Dataset before running.

**Important requirements:** explore the dataset before creating `ImageDataGenerator`; build the CNN from scratch; train for exactly **2 epochs**; use `EarlyStopping` and `ModelCheckpoint`; evaluate on test data; save the model and class mapping.

## 1. Imports and environment

In [ ]:
import os, json, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Dataset exploration — do this before creating any ImageDataGenerator
In Kaggle, add the dataset using **Add Input**. This code searches `/kaggle/input` for folders named `train`, `validation`/`valid`, and `test`. If the dataset uses different folder names, inspect the printed directory tree and update the paths manually.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = len(Path(root).parts) - len(INPUT_ROOT.parts)
    if depth <= 3:
        print("  " * depth + str(Path(root)))
        for d in dirs[:15]:
            print("  " * (depth + 1) + d + "/")

In [ ]:
# Find split directories. If auto-detection fails, set TRAIN_DIR, VAL_DIR, TEST_DIR manually.
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in image_exts)

def find_split(name_options):
    for root, dirs, files in os.walk(INPUT_ROOT):
        for d in dirs:
            if d.lower() in name_options:
                candidate = Path(root) / d
                # A split directory usually has class subdirectories.
                if any(p.is_dir() for p in candidate.iterdir()):
                    return candidate
    return None

TRAIN_DIR = find_split({"train", "training"})
VAL_DIR = find_split({"validation", "valid", "val"})
TEST_DIR = find_split({"test", "testing"})

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)

if TRAIN_DIR is None or VAL_DIR is None or TEST_DIR is None:
    raise FileNotFoundError(
        "Could not automatically find all train/validation/test folders. "
        "Inspect the directory tree above, then assign TRAIN_DIR, VAL_DIR, TEST_DIR manually."
    )

In [ ]:
def class_image_counts(split_dir):
    counts = {}
    for class_dir in sorted(p for p in Path(split_dir).iterdir() if p.is_dir()):
        counts[class_dir.name] = sum(
            1 for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        )
    return counts

train_counts = class_image_counts(TRAIN_DIR)
val_counts = class_image_counts(VAL_DIR)
test_counts = class_image_counts(TEST_DIR)

class_names = sorted(train_counts.keys())
print("Number of classes:", len(class_names))
print("Class names:", class_names)
print("Training images:", sum(train_counts.values()))
print("Validation images:", sum(val_counts.values()))
print("Testing images:", sum(test_counts.values()))
print("\nCounts by class:")
for name in class_names:
    print(f"{name:25s} train={train_counts.get(name,0):5d}  "
          f"validation={val_counts.get(name,0):5d}  test={test_counts.get(name,0):5d}")

all_counts = list(train_counts.values())
if all_counts:
    print("\nTraining distribution:",
          "approximately balanced" if max(all_counts)-min(all_counts) <= 0.2*max(1, np.mean(all_counts))
          else "possibly imbalanced; compare the class counts above")

In [ ]:
# Inspect typical image dimensions and formats
from PIL import Image
sample_files = [p for p in TRAIN_DIR.rglob("*") if p.is_file() and p.suffix.lower() in image_exts][:100]
dimensions, formats = [], {}
for p in sample_files:
    try:
        with Image.open(p) as im:
            dimensions.append(im.size)
            formats[im.format] = formats.get(im.format, 0) + 1
    except Exception:
        pass
print("Sampled image formats:", formats)
print("Sampled dimensions (width, height), first 15:", dimensions[:15])
if dimensions:
    print("Most common sampled dimensions:", __import__("collections").Counter(dimensions).most_common(5))

In [ ]:
# Display examples from different classes
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, class_name in zip(axes.flat, class_names[:8]):
    folder = Path(TRAIN_DIR) / class_name
    candidates = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]
    if candidates:
        with Image.open(candidates[0]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(class_name)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Dataset understanding notes
After reviewing the outputs above, write a short summary in your report/README: total images, class count and names, split sizes, common image dimensions/formats, folder structure, and whether the training class distribution looks balanced.

## 3. Image generators and preprocessing
Training images use random augmentation. Validation and test images are only resized and rescaled. All splits use the same class order.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True
)
eval_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_gen = train_datagen.flow_from_directory(
    str(TRAIN_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", classes=class_names, shuffle=True, seed=SEED
)
val_gen = eval_datagen.flow_from_directory(
    str(VAL_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", classes=class_names, shuffle=False
)
test_gen = eval_datagen.flow_from_directory(
    str(TEST_DIR), target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", classes=class_names, shuffle=False
)

print("Class mapping:", train_gen.class_indices)
assert train_gen.class_indices == val_gen.class_indices == test_gen.class_indices

## 4. Build a CNN from scratch
No pretrained model is used. The output layer has one unit per vegetable class.

In [ ]:
num_classes = len(class_names)

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.30),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 5. Train for exactly two epochs
`EarlyStopping(patience=3)` cannot stop before epoch 2 under normal Keras behavior; `ModelCheckpoint` saves the best validation-loss model. The `epochs` argument is exactly 2 as required.

In [ ]:
checkpoint_path = "/kaggle/working/vegetable_cnn.keras"

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)
model_checkpoint = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=2,
    callbacks=[early_stopping, model_checkpoint]
)

## 6. Evaluate on the test dataset and plot learning curves

In [ ]:
test_loss, test_accuracy = model.evaluate(test_gen, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], marker="o", label="Training accuracy")
plt.plot(history.history["val_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], marker="o", label="Training loss")
plt.plot(history.history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Show at least five test images with actual and predicted labels
test_gen.reset()
batch_images, batch_labels = next(test_gen)
predictions = model.predict(batch_images, verbose=0)
predicted_ids = np.argmax(predictions, axis=1)
actual_ids = np.argmax(batch_labels, axis=1)

n_show = min(8, len(batch_images))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat[:n_show]):
    ax.imshow(np.clip(batch_images[i], 0, 1))
    actual_name = class_names[actual_ids[i]]
    predicted_name = class_names[predicted_ids[i]]
    ax.set_title(f"Actual: {actual_name}\nPredicted: {predicted_name}")
    ax.axis("off")
for ax in axes.flat[n_show:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Save model and class mapping
Download both files from Kaggle's `/kaggle/working` output after the notebook finishes.

In [ ]:
# Save the final model and the class-label mapping used by all generators.
model.save("/kaggle/working/vegetable_cnn.keras")

class_mapping = {str(i): name for i, name in enumerate(class_names)}
with open("/kaggle/working/class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_mapping, f, ensure_ascii=False, indent=2)

# Verify model loads without retraining
loaded_model = keras.models.load_model("/kaggle/working/vegetable_cnn.keras")
print("Model loaded successfully.")
print("Saved class mapping:", class_mapping)
print("Saved files:", os.path.exists("/kaggle/working/vegetable_cnn.keras"),
      os.path.exists("/kaggle/working/class_names.json"))

## 8. Results and limitations
Record the actual test loss and accuracy printed above. Because training is limited to only two epochs, the model may be undertrained; do not claim high accuracy unless your measured results support it. Report any class imbalance or other dataset limitations you observed.